In [1]:
import jax
import jax.numpy as jnp
import jax.random as jar
from flax import struct
from typing import Any, Tuple
import chex

import malthusjax as mjx
from malthusjax.engine import GeneticEngine, GeneticEngineParams
from malthusjax.core.base import BasePopulation

print(f"JAX Version: {jax.__version__}")

JAX Version: 0.8.0


## Step 1: Defining the Custom Engine Class

We will create `CustomDiversityEngine` by inheriting from `GeneticEngine`.

We need to:
1.  Add a `diversity_weight` field to the dataclass.
2.  Implement a helper to compute crowding distance.
3.  Override `_select_parents` to use diversity scores.
4.  Override `_select_elites` to preserve diverse individuals.

**Note**: We use `@struct.dataclass` to ensure our engine remains a valid JAX PyTree, which is required for JIT compilation.

In [2]:
@struct.dataclass
class CustomDiversityEngine(GeneticEngine):
    # Add new parameters as fields
    # pytree_node=False means this is a static configuration, not a learnable parameter
    diversity_weight: float = struct.field(default=0.5, pytree_node=False)
    
    def _compute_crowding_scores(self, population: BasePopulation) -> chex.Array:
        """Helper: Compute average distance to other individuals."""
        # BasePopulation has a built-in distance_matrix method!
        # We use Hamming distance for binary genomes
        dist_matrix = population.distance_matrix(metric="hamming")
        
        # Zero out diagonal (distance to self)
        dist_matrix = dist_matrix.at[jnp.diag_indices_from(dist_matrix)].set(0.0)
        
        # Average distance to others (higher = more unique)
        return jnp.mean(dist_matrix, axis=1)

    def _select_parents(self, key: chex.Array, population: BasePopulation) -> BasePopulation:
        """Override: Select parents based on Fitness + Diversity."""
        
        # 1. Compute diversity scores
        crowding = self._compute_crowding_scores(population)
        
        # 2. Normalize scores to [0, 1] to combine them fairly
        fit_min, fit_max = jnp.min(population.fitness), jnp.max(population.fitness)
        div_min, div_max = jnp.min(crowding), jnp.max(crowding)
        
        fit_norm = (population.fitness - fit_min) / (fit_max - fit_min + 1e-8)
        div_norm = (crowding - div_min) / (div_max - div_min + 1e-8)
        
        # 3. Combine into a single score
        # diversity_weight controls the trade-off
        combined_score = (1 - self.diversity_weight) * fit_norm + self.diversity_weight * div_norm
        
        # 4. Use the standard selection operator (e.g., Tournament) with our new scores
        # The selection operator expects (key, fitness_array)
        selected_indices = self.selection(key, combined_score)
        
        return population[selected_indices]

    def _select_elites(self, population: BasePopulation, n_elites: int) -> Any:
        """Override: Preserve a mix of fittest and most diverse individuals."""
        if n_elites == 0:
            return jax.tree_util.tree_map(lambda x: x[:0], population.genes)
            
        # Split elites: 50% best fitness, 50% best diversity
        n_fit = n_elites // 2
        n_div = n_elites - n_fit
        
        # Select top fitness elites
        _, fit_indices = jax.lax.top_k(population.fitness, n_fit)
        
        # Select top diversity elites
        crowding = self._compute_crowding_scores(population)
        _, div_indices = jax.lax.top_k(crowding, n_div)
        
        # Combine indices
        all_indices = jnp.concatenate([fit_indices, div_indices])
        
        # Return the GENES (not the population object)
        return population[all_indices].genes

## Step 2: Setting up the Experiment

We'll use a standard Binary Optimization problem (OneMax) to test our engine.

In [3]:
# 1. Define Problem
genome_conf = mjx.BinaryGenomeConfig(length=100)
evaluator = mjx.BinarySumEvaluator(mjx.BinarySumConfig(maximize=True))

# 2. Define Operators
# Note: We use standard operators from Level 2
selection = mjx.selection.Tournament(num_selections=100, tournament_size=3)
crossover = mjx.crossover.Uniform(num_offspring=1, crossover_rate=0.8)
mutation = mjx.mutation.BitFlip(num_offspring=1, mutation_rate=0.01)

# 3. Instantiate our Custom Engine
# We set diversity_weight=0.5 for a balanced approach
engine = CustomDiversityEngine(
    genome_config=genome_conf,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation,
    diversity_weight=0.5
)

print("Custom Engine instantiated.")

Custom Engine instantiated.


## Step 3: Running the Evolution

We run the engine just like the standard `GeneticEngine`. The `run()` method handles the JIT compilation loop automatically.

In [4]:
# Configure Run Parameters
params = GeneticEngineParams(
    pop_size=100,
    num_generations=50,
    elitism=4
)

# Initialize State
key = jar.PRNGKey(42)
key, k_init = jar.split(key)
state = engine.init_state(k_init, params)

print(f"Initial Best Fitness: {state.best_fitness}")

# Run Evolution
# compile=True enables JIT compilation via jax.lax.scan
final_state, history, elapsed_time = engine.run(
    state, 
    params, 
    time_it=True, 
    compile=True, 
    verbose=False
)

print(f"Evolution finished in {elapsed_time:.4f}s")
print(f"Final Best Fitness: {final_state.best_fitness}")

Initial Best Fitness: 67.0
Evolution finished in 0.5186s
Final Best Fitness: 87.0
Evolution finished in 0.5186s
Final Best Fitness: 87.0


## Step 4: Analyzing Diversity

Let's verify that our engine actually maintained diversity. We can check the final population's distance matrix.

In [5]:
# Compute final diversity (Average Hamming Distance)
final_dist_matrix = final_state.population.distance_matrix(metric="hamming")
# Exclude diagonal
final_dist_matrix = final_dist_matrix.at[jnp.diag_indices_from(final_dist_matrix)].set(0.0)
avg_diversity = jnp.mean(final_dist_matrix)

print(f"Final Population Diversity: {avg_diversity:.4f}")
print(f"Theoretical Max Diversity (approx): {genome_conf.length * 0.5}") # For random binary strings

Final Population Diversity: 38.9924
Theoretical Max Diversity (approx): 50.0


## Conclusion

By overriding just two methods (`_select_parents` and `_select_elites`), we completely altered the evolutionary pressure of the algorithm to favor diversity.

This **Template Method** approach allows you to implement:
- **Novelty Search**: By replacing fitness with novelty scores.
- **Island Models**: By modifying `_select_parents` to restrict mating to sub-populations.
- **Age-Layered Evolution**: By modifying `_select_elites` to respect age constraints.

All while keeping the high-performance JAX loop intact!